# 03 - Train/val split strategy

`Train.csv`/`Test.csv` have no document/page column (see `notebooks/01_eda_text_overview.ipynb`), so the group-aware split the README originally called for can't be keyed on a real document ID. The only entity-like signal available is exact-duplicate `Target` text (10 of 4,086 unique lines repeat, up to 3x - common legal boilerplate phrases, e.g. "Sealed and delivered in the presence of"). This notebook compares a plain random split against a split grouped on exact `Target` text, using [`GroupShuffleSplit`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupShuffleSplit.html) so identical lines can never land on both sides.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

from src.config import RANDOM_STATE, TARGET_COLUMN
from src.data import load_raw_data

train = load_raw_data('train')
train['len'] = train[TARGET_COLUMN].str.len()

dup_targets = train[TARGET_COLUMN].value_counts()
dup_targets = dup_targets[dup_targets > 1].index
len(dup_targets)

10

In [2]:
def duplicate_leakage(fit_df, val_df, dup_targets, target_col=TARGET_COLUMN):
    """Count duplicate-text groups that end up split across both sides."""
    leaked = 0
    for t in dup_targets:
        in_fit = (fit_df[target_col] == t).any()
        in_val = (val_df[target_col] == t).any()
        if in_fit and in_val:
            leaked += 1
    return leaked

In [3]:
# Baseline: plain random split (what src/train.py currently does)
fit_rand, val_rand = train_test_split(train, test_size=0.2, random_state=RANDOM_STATE)

print('sizes:', len(fit_rand), len(val_rand))
print('len mean/std - fit:', fit_rand['len'].mean(), fit_rand['len'].std())
print('len mean/std - val:', val_rand['len'].mean(), val_rand['len'].std())
print('duplicate groups leaked across fit/val:', duplicate_leakage(fit_rand, val_rand, dup_targets), '/', len(dup_targets))

sizes: 3278 820
len mean/std - fit: 62.25137278828554 12.649941336385407
len mean/std - val: 62.385365853658534 12.934238874979203
duplicate groups leaked across fit/val: 3 / 10


In [4]:
# Candidate: group split keyed on exact Target text
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
fit_idx, val_idx = next(gss.split(train, groups=train[TARGET_COLUMN]))
fit_grp, val_grp = train.iloc[fit_idx], train.iloc[val_idx]

print('sizes:', len(fit_grp), len(val_grp))
print('len mean/std - fit:', fit_grp['len'].mean(), fit_grp['len'].std())
print('len mean/std - val:', val_grp['len'].mean(), val_grp['len'].std())
print('duplicate groups leaked across fit/val:', duplicate_leakage(fit_grp, val_grp, dup_targets), '/', len(dup_targets))

sizes: 3280 818
len mean/std - fit: 62.270426829268295 12.731672946067347
len mean/std - val: 62.30929095354523 12.609508049653954
duplicate groups leaked across fit/val: 0 / 10


## Decision

Plain random split leaks 3/10 duplicate-text groups across fit/val; the `GroupShuffleSplit` version leaks 0, keeps essentially the same ~80/20 size ratio, and matches the plain split's length distribution almost exactly (no distortion from grouping). Adopting the group-aware split as `src/data.py::split_train_val` and wiring it into `src/train.py`, replacing the plain `train_test_split` call.